## 0. Описание лабораторной работы

**Задание:**

- Взять любой датасет классификации текста; сделать базовое описание выбранного датасета.
- Реализовать следующие компоненты с нуля:
  - **MultiHeadAttention**
  - **PositionalEncoding** (на основе косинусных функций)
  - **TransformerEncoderLayer**
  - **TransformerEncoder**
  - **TransformerClassifier**

- Реализовать пайплайн обучения и обучить модель классификации текста.
- Продемонстрировать результаты (точность, потери, примеры предсказаний).

- **Можно использовать:**
  - Готовый токенизатор
  - Базовые слои PyTorch
  - Оптимизатор
  - Функции метрик

- **Нельзя использовать:**
  - Готовый текстовый эмбеддер
  - Готовый SDPA
  - Готовые блоки Transformer

In [74]:
import warnings
warnings.filterwarnings('ignore')
import plotly.io as pio
import plotly.express as px
pio.templates.default = 'plotly_white'
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from PIL import Image

import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import random
import sklearn
import torch
from torch import nn
import os
import io

seed = 42
random.seed(seed)
np.random.seed(seed)
sklearn.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

## 1. Подготовка данных


**Цель:** Создать модель машинного обучения для бинарной классификации тональности отзывов о фильмах (негативный/позитивный) на основе **полных текстов рецензий** (text)

**Текстовые данные** (100% оригинальных отзывов):
   - Формат: Полные текстовые рецензии на английском языке
   - Длина: Развернутые отзывы существенной длины
   - Контент: Аргументированная критика или похвала фильмов

**Целевая переменная:**
- **Sentiment**: бинарная классификация (neg / pos или 0 / 1)

In [60]:
from datasets import load_dataset
dataset = load_dataset("stanfordnlp/imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [72]:
def visualize_sentiment_samples(dataset_split, n_per_sentiment=5, max_chars=200):
    neg_samples = []
    pos_samples = []

    for sample in dataset_split:
        label = sample.get('label', None)
        if label == 0 and len(neg_samples) < n_per_sentiment:
            neg_samples.append(sample)
        elif label == 1 and len(pos_samples) < n_per_sentiment:
            pos_samples.append(sample)

        if len(neg_samples) >= n_per_sentiment and len(pos_samples) >= n_per_sentiment:
            break

    all_samples = neg_samples + pos_samples

    for idx, sample in enumerate(all_samples):
        label = sample.get('label', -1)
        text = sample.get('text', '')
        sentiment = "Negative" if label == 0 else "Positive" if label == 1 else "Unknown"

        print(f"\n{'='*60}")
        print(f"Sample {idx + 1}: {sentiment} (label: {label})")
        print(f"{'='*60}")

        if text:
            display_text = text[:max_chars] + ("..." if len(text) > max_chars else "")
            print(f"{display_text}")
            print(f"\nLength: {len(text)} chars")

In [73]:
visualize_sentiment_samples(dataset['train'])


Sample 1: Negative (label: 0)
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev...

Length: 1640 chars

Sample 2: Negative (label: 0)
"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that ...

Length: 1294 chars

Sample 3: Negative (label: 0)
If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches ...

Length: 528 chars

Sample 4: Negative (label: 0)
This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impre...



In [107]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def collate_batch(batch):
    texts = [item['text'] for item in batch]
    labels = [item['label'] for item in batch]

    encoded = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"
    )

    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'label': torch.tensor(labels)
    }

train_loader = DataLoader(
    dataset['train'],
    batch_size=32,
    shuffle=True,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    dataset['test'],
    batch_size=32,
    shuffle=False,
    collate_fn=collate_batch
)

## 2. Реализации attention

Реализация синусоидального позиционного кодирования для трансформеров.
    
Класс добавляет информацию о позиции токенов в последовательности к их эмбеддингам. Используется синусоидальная функция для четных индексов и косинусоидальная для нечетных, что позволяет модели обучать относительные позиции.
    
Основные характеристики:
- Позиционные эмбеддинг зависят от позиции токена в последовательности
- Частота кодирования уменьшается экспоненциально по размерности эмбеддинга
- Поддерживаются последовательности переменной длины с кэшированием

Используется формула из оригинальной статьи "Attention Is All You Need":
$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad\quad
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

In [85]:
class CustomPositionalEncoding(nn.Module):
    def __init__(self, embedding_dim, max_sequence_length=512, base_frequency=10000.0):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.base_frequency = base_frequency
        self.max_sequence_length = max_sequence_length
        self._cached_encodings = {}

    def _compute_encoding(self, sequence_length):
        if sequence_length not in self._cached_encodings:
            position_indices = torch.arange(sequence_length).unsqueeze(1)

            frequency_scaling = torch.exp(
                torch.arange(0, self.embedding_dim, 2) *
                (-torch.log(torch.tensor(self.base_frequency)) / self.embedding_dim)
            )

            encoding = torch.zeros(sequence_length, self.embedding_dim)
            encoding[:, 0::2] = torch.sin(position_indices * frequency_scaling)
            encoding[:, 1::2] = torch.cos(position_indices * frequency_scaling)
            self._cached_encodings[sequence_length] = encoding

        return self._cached_encodings[sequence_length]

    def forward(self, token_embeddings):
        sequence_length = token_embeddings.size(1)

        if sequence_length > self.max_sequence_length:
            raise ValueError(f"Sequence length {sequence_length} is greater than limit {self.max_sequence_length}")

        encoding = self._compute_encoding(sequence_length)
        encoding_with_batch = encoding.unsqueeze(0).to(token_embeddings.device)
        return token_embeddings + encoding_with_batch

Формула Multi-Head Attention:
$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O
$$

$$
\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

Обозначения:
- $ Q, K, V $ — матрицы запросов (queries), ключей (keys) и значений (values)
- $ d_k $ — размерность ключей (head_dimension)
- $ h $ — количество голов механизма внимания (num_heads)
- $ W_i^Q, W_i^K, W_i^V $ — матрицы проекции для i-й головы
- $ W^O $ — итоговая матрица проекции


Масштабирование на $ \sqrt{d_k} $ предотвращает слишком маленькие градиенты

In [94]:
class CustomMultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.head_dimension = hidden_size // num_heads
        self.num_heads = num_heads
        self.hidden_size = hidden_size
        self.query_projection = nn.Linear(hidden_size, hidden_size)
        self.key_projection = nn.Linear(hidden_size, hidden_size)
        self.value_projection = nn.Linear(hidden_size, hidden_size)
        self.output_projection = nn.Linear(hidden_size, hidden_size)

    def _reshape_for_attention(self, tensor):
        batch_size, sequence_length, hidden_size = tensor.shape
        return tensor.reshape(batch_size, sequence_length, self.num_heads, self.head_dimension).transpose(1, 2)

    def forward(self, input_sequence, attention_mask=None):
        batch_size, sequence_length, hidden_size = input_sequence.size()

        queries = self._reshape_for_attention(self.query_projection(input_sequence))
        keys = self._reshape_for_attention(self.key_projection(input_sequence))
        values = self._reshape_for_attention(self.value_projection(input_sequence))

        attention_scores = torch.matmul(queries, keys.transpose(-2, -1))
        attention_scores = attention_scores / torch.sqrt(torch.tensor(self.head_dimension, dtype=torch.float32))

        if attention_mask is not None:
            attention_scores = attention_scores.masked_fill(attention_mask[:, None, None, :] == 0, float('-inf'))

        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_output = torch.matmul(attention_weights, values)
        attention_output = attention_output.transpose(1, 2)
        attention_output = attention_output.reshape(batch_size, sequence_length, hidden_size)
        return self.output_projection(attention_output)

### Transformer Encoder Layer

**Переменные:**
- `hidden_size` : Размерность эмбеддингов токенов (например, 512, 768)
- `num_heads`: Количество голов внимания в multi-head attention
- `feedforward_dimension`: Размерность скрытого слоя в feed-forward сети (обычно 4×hidden_size)
- `dropout_rate`: Вероятность dropout для регуляризации

**Компоненты слоя:**
1. Self Attention Block:
   - `self_attention`: Multi-head self-attention механизм
   - Вычисляет зависимости между всеми токенами последовательности

2. Feed-forward Network:
   - `feedforward_network`: Двухслойный персептрон с ReLU активацией
   - Применяется к каждому токену независимо

3. Layer Normalization:
   - `pre_attention_layer_norm`: Нормализация перед блоком внимания
   - `pre_feedforward_layer_norm`: Нормализация перед feed-forward сетью

4. Residual Connections:
   - Добавление выхода каждого блока к его входу
   - Помогает градиентам распространяться в глубоких сетях

In [91]:
class CustomTransformerEncoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, feedforward_dimension, dropout_rate=0.1):
        super().__init__()

        self.self_attention_block = nn.ModuleDict({
            'attention': CustomMultiHeadAttention(hidden_size, num_heads),
            'layer_norm': nn.LayerNorm(hidden_size),
            'dropout': nn.Dropout(dropout_rate)
        })

        self.feedforward_block = nn.ModuleDict({
            'linear_1': nn.Linear(hidden_size, feedforward_dimension),
            'activation': nn.ReLU(),
            'linear_2': nn.Linear(feedforward_dimension, hidden_size),
            'layer_norm': nn.LayerNorm(hidden_size),
            'dropout': nn.Dropout(dropout_rate)
        })

    def _apply_residual_block(self, input_tensor, block_output, layer_norm, dropout):
        residual_connection = input_tensor + dropout(block_output)
        return layer_norm(residual_connection)

    def forward(self, input_sequence, attention_mask=None):
        attention_output = self.self_attention_block['attention'](input_sequence, attention_mask)
        normalized_attention_output = self._apply_residual_block(
            input_sequence,
            attention_output,
            self.self_attention_block['layer_norm'],
            self.self_attention_block['dropout']
        )

        feedforward_hidden = self.feedforward_block['linear_1'](normalized_attention_output)
        feedforward_hidden = self.feedforward_block['activation'](feedforward_hidden)
        feedforward_output = self.feedforward_block['linear_2'](feedforward_hidden)

        final_output = self._apply_residual_block(
            normalized_attention_output,
            feedforward_output,
            self.feedforward_block['layer_norm'],
            self.feedforward_block['dropout']
        )

        return final_output

### TransformerEncoder

**Назначение:**
Стек из нескольких идентичных слоев Transformer Encoder для последовательной обработки входной последовательности.

**Аргументы конструктора:**
- `num_layers` (int): Количество слоев энкодера в стеке (обычно 6-12 для базовых моделей)
- `hidden_size` (int): Размерность эмбеддингов (например, 512, 768)
- `num_heads` (int): Количество голов внимания в каждом слое
- `feedforward_dimension` (int): Размерность скрытого слоя в feed-forward сети
- `dropout_rate` (float, optional=0.1): Вероятность dropout для регуляризации
- `apply_final_norm` (bool, optional=True): Применять ли финальную LayerNorm

**Основные компоненты:**
1. **Стек слоев энкодера** (`encoder_layers`):
   - Список из `num_layers` идентичных `CustomTransformerEncoderLayer`
   - Каждый слой содержит self-attention и feed-forward блоки

2. **Финальная нормализация** (`final_layer_norm`):
   - Опциональная LayerNorm после всех слоев
   - Стабилизирует выход для следующих компонентов модели

**Forward метод:**
- `input_sequence`: Входные эмбеддинги токенов `[batch_size, sequence_length, hidden_size]`
- `attention_mask`: Маска внимания для padding или causal masking
- `return_all_layers`: Возвращать выходы всех слоев или только последнего

**Режимы работы:**
1. **Обычный режим** (`return_all_layers=False`):
   - Возвращает только выход последнего слоя
   - Эффективно по памяти

2. **Расширенный режим** (`return_all_layers=True`):
   - Возвращает кортеж `(final_output, layer_outputs)`
   - `layer_outputs` содержит выходы каждого слоя
   - Полезно для анализа или промежуточных представлений

In [105]:
class CustomTransformerEncoder(nn.Module):
    def __init__(self, num_layers, hidden_size, num_heads, feedforward_dimension,
                 dropout_rate=0.1, apply_final_norm=True):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.apply_final_norm = apply_final_norm

        self.encoder_layers = nn.ModuleList([
            CustomTransformerEncoderLayer(
                hidden_size=hidden_size,
                num_heads=num_heads,
                feedforward_dimension=feedforward_dimension,
                dropout_rate=dropout_rate
            )
            for layer_idx in range(num_layers)
        ])

        if apply_final_norm:
            self.final_layer_norm = nn.LayerNorm(hidden_size)

    def forward(self, input_sequence, attention_mask=None, return_all_layers=False):
        if return_all_layers:
            layer_outputs = []
            current_sequence = input_sequence

            for layer_idx, encoder_layer in enumerate(self.encoder_layers):
                current_sequence = encoder_layer(current_sequence, attention_mask)
                layer_outputs.append(current_sequence)

            if self.apply_final_norm:
                current_sequence = self.final_layer_norm(current_sequence)
                layer_outputs[-1] = current_sequence

            return current_sequence, layer_outputs
        else:
            encoder_output = input_sequence

            for encoder_layer in self.encoder_layers:
                encoder_output = encoder_layer(encoder_output, attention_mask)

            if self.apply_final_norm:
                encoder_output = self.final_layer_norm(encoder_output)

            return encoder_output

In [93]:
class CustomTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, emb_dim=256,
                 num_layers=6, num_heads=8, ff_dim=1024, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.pos = CustomPositionalEncoding(emb_dim)
        self.encoder = CustomTransformerEncoder(num_layers=num_layers,
                                                hidden_size=emb_dim,
                                                num_heads=num_heads,
                                                feedforward_dimension=ff_dim,
                                                dropout_rate=dropout)
        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_dim))
        self.head = nn.Linear(emb_dim, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: [batch, seq_len]
        emb = self.emb(x)  # [batch, seq_len, emb_dim]
        # add CLS token
        batch_size = x.size(0)
        cls = self.cls_token.expand(batch_size, -1, -1)
        emb = torch.cat([cls, emb], dim=1)  # [batch, seq_len+1, emb_dim]

        if mask is not None:
            cls_mask = torch.ones(batch_size, 1, device=mask.device)
            mask = torch.cat([cls_mask, mask], dim=1)  # [batch, seq_len+1]
        emb = self.pos(emb)
        encoded = self.encoder(emb, mask)
        cls_out = encoded[:, 0, :]  # [batch, emb_dim]
        cls_out = self.dropout(cls_out)
        return self.head(cls_out)  # [batch, num_classes]

## Обучение

In [108]:
from tqdm import tqdm

def train_custom_transformer_model(train_loader, test_loader, tokenizer, num_classes, num_epochs=3, batch_size=32, learning_rate=3e-4):
    device = 'cpu'

    criterion = nn.CrossEntropyLoss()

    model = CustomTransformerClassifier(
        vocab_size=tokenizer.vocab_size,
        num_classes=num_classes,
        emb_dim=256,
        num_layers=6,
        num_heads=8,
        ff_dim=1024,
        dropout=0.1
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for batch in train_pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            # forward
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            # backward
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            predictions = logits.argmax(dim=1)
            train_correct += (predictions == labels).sum().item()
            train_total += labels.size(0)

            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{(predictions == labels).float().mean().item():.4f}'
            })

        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = train_correct / train_total

        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            test_pbar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Test]")
            for batch in test_pbar:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)

                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                test_loss += loss.item()
                predictions = logits.argmax(dim=1)
                test_correct += (predictions == labels).sum().item()
                test_total += labels.size(0)

                test_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{(predictions == labels).float().mean().item():.4f}'
                })

        avg_test_loss = test_loss / len(test_loader)
        test_accuracy = test_correct / test_total

        print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
        print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.4f}")
        print(f"  Test Loss:  {avg_test_loss:.4f} | Test Acc:  {test_accuracy:.4f}")
        print("-" * 50)

    return model

In [ ]:
model = train_custom_transformer_model(train_loader, test_loader, tokenizer, num_classes=2)

Epoch 1/3 [Train]:   4%|▍         | 30/782 [04:26<2:33:35, 12.26s/it, loss=0.6996, acc=0.6250]

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

def evaluate_model(model, test_loader, tokenizer):
    device = "cpu"
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    all_logits = []
    all_texts = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_logits.extend(logits.cpu().numpy())

            for ids in input_ids.cpu().numpy():
                tokens = tokenizer.convert_ids_to_tokens(ids[:15])
                text_preview = tokenizer.convert_tokens_to_string(tokens)
                all_texts.append(text_preview)

    f1 = f1_score(all_labels, all_preds, average="weighted")
    acc = accuracy_score(all_labels, all_preds)

    print(f"F1-score: {f1:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"\nClassification Report:\n{classification_report(all_labels, all_preds)}")

    create_plots(all_labels, all_preds, all_logits, all_texts)

    return all_preds, all_labels, f1, acc

In [ ]:
predictions, labels, f1, acc = evaluate_model(model, test_loader, tokenizer)

In [ ]:
# @title visualize metrics
def create_plots(true_labels, predictions, texts):
    cm = confusion_matrix(true_labels, predictions)

    fig_cm = go.Figure(data=go.Heatmap(
        z=cm,
        x=['Negative', 'Positive'],
        y=['Negative', 'Positive'],
        colorscale='Blues'
    ))
    fig_cm.update_layout(title='Confusion Matrix')

    df = pd.DataFrame({
        'True': true_labels,
        'Predicted': predictions,
        'Correct': [t == p for t, p in zip(true_labels, predictions)],
        'Text': texts[:len(true_labels)],
    })

    fig_conf = px.histogram(df, x='Confidence', color='Correct',
                           barmode='overlay', title='Confidence Distribution')

    fig_dist = make_subplots(rows=1, cols=2,
                           subplot_titles=['True Distribution', 'Predicted Distribution'])
    fig_dist.add_trace(go.Bar(x=['Negative', 'Positive'],
                            y=[sum(np.array(true_labels) == 0), sum(np.array(true_labels) == 1)]), row=1, col=1)
    fig_dist.add_trace(go.Bar(x=['Negative', 'Positive'],
                            y=[sum(np.array(predictions) == 0), sum(np.array(predictions) == 1)]), row=1, col=2)

    interesting_samples = df[df['Correct'] == False].head(5)
    if len(interesting_samples) > 0:
        fig_table = go.Figure(data=[go.Table(
            header=dict(values=['Text Snippet', 'True', 'Predicted', 'Confidence']),
            cells=dict(values=[
                interesting_samples['Text'].str[:100] + '...',
                interesting_samples['True'].map({0: 'Negative', 1: 'Positive'}),
                interesting_samples['Predicted'].map({0: 'Negative', 1: 'Positive'}),
                interesting_samples['Confidence'].round(3)
            ])
        )])
        fig_table.update_layout(title='Misclassified Examples')
    else:
        fig_table = None

    fig_cm.show()
    fig_conf.show()
    fig_dist.show()
    if fig_table:
        fig_table.show()

In [ ]:
def extract_texts(test_loader, tokenizer):
    texts = []
    for batch in test_loader:
        for seq_ids in batch["input_ids"]:
            text = tokenizer.decode(seq_ids, skip_special_tokens=True)
            texts.append(text)
    return texts

all_texts = extract_texts(test_loader, tokenizer)
create_plots(true_labels=labels, predictions=preictions, texts=all_texts)

In [ ]:
from datetime import datetime
from dateutil import tz

current_time = datetime.now(tz.gettz('Etc/GMT-3'))
print(f"Current time in UTC+3: {current_time}")

**Лабораторная работа выполнена в рамках курса "Машинное обучение" ИТМО, 2025**